<a href="https://colab.research.google.com/github/Keerthan-AC-583/BIS-Lab/blob/main/1BM24CS133_ACO_University_Time_Table.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


import numpy as np
import random

exams = ["Maths", "Physics", "Chemistry", "CS", "OS"]

students = {
    "Maths": {"S1", "S2", "S3", "S4"},
    "Physics": {"S1", "S5", "S6"},
    "Chemistry": {"S2", "S5", "S7"},
    "CS": {"S3", "S6", "S8"},
    "OS": {"S4", "S7", "S8"}
}

slots = ["9 AM", "1 PM", "4 PM"]

halls = {
    "Hall A": 30,
    "Hall B": 25,
    "Hall C": 20
}

exam_size = {
    exam: len(students[exam]) for exam in exams
}

n_ants = 10
alpha = 1
beta = 2
rho = 0.5
Q = 100
MaxIter = 50

n_exams = len(exams)
n_slots = len(slots)
n_halls = len(halls)

pheromone = np.ones(
    (n_exams, n_slots, n_halls)
)

heuristic = np.ones(
    (n_exams, n_slots, n_halls)
)

conflict_degree = {}

for exam in exams:
    conflict_degree[exam] = sum(
        bool(students[exam] & students[other])
        for other in exams if other != exam
    )

exam_order = sorted(
    exams,
    key=lambda x: conflict_degree[x],
    reverse=True
)

def is_feasible(exam, slot, hall, timetable):

    if exam_size[exam] > halls[hall]:
        return False

    for other, (other_slot, other_hall) in timetable.items():

        if slot == other_slot and hall == other_hall:
            return False

        if slot == other_slot:
            if students[exam] & students[other]:
                return False

    return True


def calculate_fitness(timetable):

    used_slots = len(set(
        slot for slot, hall in timetable.values()
    ))

    slot_counts = [
        sum(1 for slot, hall in timetable.values()
            if slot == s)
        for s in range(n_slots)
    ]

    fitness = (
        used_slots * 100 +
        0.1 * sum(count ** 2 for count in slot_counts)
    )

    return fitness


def construct_solution():

    timetable = {}

    for exam in exam_order:

        candidates = []

        probabilities = []

        for s in range(n_slots):
            for h in range(n_halls):

                hall = list(halls.keys())[h]

                if is_feasible(exam, s, hall, timetable):

                    candidates.append((s, h))

                    tau = pheromone[
                        exams.index(exam), s, h
                    ]

                    eta = heuristic[
                        exams.index(exam), s, h
                    ]

                    weight = (tau ** alpha) * (eta ** beta)

                    probabilities.append(weight)

        if not candidates:
            return None

        total = sum(probabilities)

        probabilities = [
            p / total for p in probabilities
        ]

        selected = random.choices(
            candidates,
            weights=probabilities,
            k=1
        )[0]

        s, h = selected

        timetable[exam] = (s, list(halls.keys())[h])

    return timetable


best_timetable = None
best_fitness = float("inf")

for iteration in range(MaxIter):

    ant_solutions = []

    for ant in range(n_ants):

        timetable = construct_solution()

        if timetable is not None:

            fitness = calculate_fitness(timetable)

            ant_solutions.append((timetable, fitness))

            if fitness < best_fitness:

                best_fitness = fitness
                best_timetable = timetable.copy()

    pheromone *= (1 - rho)

    for timetable, fitness in ant_solutions:

        deposit = Q / fitness

        for exam, (s, hall) in timetable.items():

            i = exams.index(exam)
            h = list(halls.keys()).index(hall)

            pheromone[i, s, h] += deposit

    if (iteration + 1) % 10 == 0:
        print(
            "Iteration:", iteration + 1,
            "Best Fitness:", round(best_fitness, 2)
        )


print("\n===== BEST EXAMINATION TIMETABLE =====")

if best_timetable is not None:

    for exam in exams:

        s, hall = best_timetable[exam]

        print(
            exam,
            " | Time:", slots[s],
            " | Hall:", hall
        )

    print("\nBest Fitness:", round(best_fitness, 2))

    print("Total Conflicts: 0")

    print("Optimization Completed!")

else:
    print("No feasible timetable found.")




Iteration: 10 Best Fitness: 300.9
Iteration: 20 Best Fitness: 300.9
Iteration: 30 Best Fitness: 300.9
Iteration: 40 Best Fitness: 300.9
Iteration: 50 Best Fitness: 300.9

===== BEST EXAMINATION TIMETABLE =====
Maths  | Time: 4 PM  | Hall: Hall B
Physics  | Time: 1 PM  | Hall: Hall C
Chemistry  | Time: 9 AM  | Hall: Hall B
CS  | Time: 9 AM  | Hall: Hall A
OS  | Time: 1 PM  | Hall: Hall B

Best Fitness: 300.9
Total Conflicts: 0
Optimization Completed!
